# WarpTracer — vehicle-mounted LiDAR

One box car, one floor, four walls, with LiDAR mounted above the car.
Driving uses the lean flat-ground model; drop/impact use Newton.
The default disparity extender steers and brakes using the mounted LiDAR. Scripted driving checks are also available.
Yellow points show returns; green lines show a sparse selection of laser rays.
All 108 returns are saved and displayed; ray lines use a sparse subset.

Run the three cells below. Setup reuses `/content/WarpTracer-rigidbody` and its
single root uv environment, switches to `performance-prototype`, and updates that branch.
Choose a GPU runtime to use CUDA; the demo also works on CPU. The first run compiles
Warp kernels. Orbit, zoom, pause, and scrub the recorded replay.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "performance-prototype", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "performance-prototype:refs/remotes/origin/performance-prototype"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/performance-prototype"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "performance-prototype", "origin/performance-prototype"], check=True)
subprocess.run(["git", "switch", "performance-prototype"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "performance-prototype"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked"], check=True)

def run_stream(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except KeyboardInterrupt:
            import signal
            os.killpg(process.pid, signal.SIGTERM)
            process.wait()
            raise


## Choose a driving check

- `"disparity"`: navigate from LiDAR, with a 1.5 m/s speed limit.
- `"accelerate-brake"`: accelerate straight, then brake to a stop.
- `"circle"`: drive with a constant left steering angle.
- `"s-turn"`: alternate left/right steering, then stop.

Change the scenario and rerun the final two cells. The existing `"drop"` and
`"wall-impact"` checks are also available. Physics runs at 240 Hz; the viewer
records 30 poses per second. Tire support currently assumes a flat floor.

This notebook uses LiDAR at 60 Hz with a 270-degree field of view and a 30 m maximum range.
Disparity navigation consumes a scan, chooses controls, advances physics, then scans again.
Range arrays, validity masks, timestamps, and sensor poses are saved in the NPZ.


In [ ]:
scenario = "disparity"  # or "accelerate-brake", "circle", "s-turn"
run_stream(["uv", "run", "warptracer", "demo", scenario])


In [ ]:
from html import escape
from IPython.display import HTML, display

replay = Path("outputs") / f"{scenario}.html"
display(HTML(f'<iframe srcdoc="{escape(replay.read_text(), quote=True)}" '
             'width="100%" height="650" style="border:0"></iframe>'))
